# Experiment 3: CPU Core Pressure Benchmark

This experiment evaluates the performance of the OCR + Mistral inference
pipeline under different CPU core limits.

- Electron / Flask UI is intentionally skipped
- Only the core pipeline is benchmarked
- One Docker run = one CPU limit
- Output: CSV file per CPU configuration
CPU Scaling & Latency Benchmarking

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


Import Required Modules

In [ ]:
import os
import time
import random
import pandas as pd

from model.model_implement import run_pipeline_for_experiment

## Read CPU Limit from Environment

The CPU limit is passed when launching the Docker container.
This value is recorded in the results for analysis.


In [ ]:
cpu_cores = os.environ.get("CPU_CORES", "1.0")
print(f"Running experiment with CPU limit = {cpu_cores} cores")

## Dataset Sampling

We randomly sample a fixed number of PDFs to ensure consistency
across different CPU configurations.
Dataset Sampling

In [ ]:
REPORTS_DIR = "../report/reports"

pdf_files = [
    os.path.join(root, f)
    for root, _, files in os.walk(REPORTS_DIR)
    for f in files if f.endswith(".pdf")
]

print(f"Total PDFs found: {len(pdf_files)}")

random.seed(42)
SELECTED_PDFS = random.sample(pdf_files, min(10, len(pdf_files)))


## Experiment Runner

Each PDF is processed independently.
Latency and success/failure status are recorded.


In [ ]:
def run_cpu_capped_experiment(cpu_limit, pdf_batch):
    results = []

    for pdf in pdf_batch:
        start_time = time.time()
        status = "success"
        error_type = None

        try:
            result = run_pipeline_for_experiment(
                pdf_path=pdf,
                language="English",
                model="mistral"
            )
            latency = time.time() - start_time

            if result is None or "error" in result:
                status = "failure"
                error_type = result.get("error") if result else "Unknown error"

        except Exception as e:
            latency = time.time() - start_time
            status = "failure"
            error_type = str(e)

        results.append({
            "pdf_name": os.path.basename(pdf),
            "cpu_cores": float(cpu_limit),
            "status": status,
            "latency_sec": latency,
            "error_type": error_type
        })

    return results


## Run Experiment and Save Results


In [ ]:
results = run_cpu_capped_experiment(cpu_cores, SELECTED_PDFS)

df = pd.DataFrame(results)
output_file = f"cpu_pressure_results_{cpu_cores}CORES.csv"

df.to_csv(output_file, index=False)
print(f"Results saved to {output_file}")
